# Dynamiczny dobór narzędzi

## Cel: Zaprojektowanie i implementacja adaptacyjnego mechanizmu doboru narzędzi z wykorzystaniem dynamicznych modeli danych.

### Czego się nauczysz:
1. Jak zaimplementować warstwę *Discovery* oddzieloną od warstwy *Execution*?
2. Dlaczego warto ograniczać pulę narzędzi dostępnych dla modelu?
3. Jak wykorzystać *Resource Memory* do przechowywania stanu narzędzi?
4. W jaki sposób typ Union wymusza na modelu konkretne decyzje?
5. Jak dynamicznie tworzyć klasy Pydantic w trakcie działania programu?
6. Dlaczego strategie *Search & Call* są stabilniejsze od wrzucania wszystkiego do kontekstu?
7. Jak zaprojektować router wstępnie przetwarzający intencje użytkownika?



## Robimy "drugie kółko" ✈️

Właściwie o wszystkich fundamentalnych klockach już rozmawialiśmy w poprzednich lekcjach tego modułu (i w module o pamięci). Wiesz już, dlaczego "naiwne podejście" (wrzucanie wszystkiego do worka) prowadzi do katastrofy i zatruwa kontekst. Wiesz też, że magiczne tryby typu "Code Mode" to tylko plaster, a nie rozwiązanie problemu u podstaw.

W tej lekcji robimy to, co piloci nazywają "drugim kręgiem" lub podejściem na drugie kółko.

Dlaczego? Bo najlepiej uczyć się na przykładach, które stopniowo odsłaniają kolejne warstwy złożoności. Wcześniej skupialiśmy się na tym **CO** (definicje), teraz skupimy się na **JAK** (architektura). Chcę, żebyśmy te same klocki, które już znasz, poukładali w nieco inny sposób, budując bardziej zaawansowany system.

Popatrzmy jeszcze raz na nasz **Spektrum Autonomii AI**.

![](../../module1/images/software3.0_7_levels.jpg)

Ból polega na tym, że wizja MCP, która zakładam, że Agent AI "sam wie", co zrobić, leży na **Poziomie 6 lub 7**. Tam systemy "automagicznie" odnajdują narzędzia. Jednak technologicznie, jeśli chodzi o realizację, twardo stąpamy po **poziomie 4** (teraz mówimy z perspektywy tego poziomu). Próba realizowania wizji z poziomu 7 narzędziami z poziomu 1 lub 2 kończy się frustracją (np. code mode to jest właśnie poziom 1 ewentualnie poziom 2).

Dlatego zamiast udawać, że magia istnieje, zastosujemy inżynierię.



## Szybkie przypomnienie kontekstu

Aby to miało ręce i nogi, musimy przypomnieć sobie trzy obszary odpowiedzialności, o których wspominaliśmy. Ale tym razem spójrz na nie przez pryzmat **Resource Management Memory** (pamięci zasobów), którą omawialiśmy w Module 2.

Dla przypomnienia warstwy pamięci:

1.  🏛️ Constitutional Memory – "DNA agenta". Zasady bezpieczeństwa i etyki, których agentowi **NIE WOLNO** złamać. To tutaj są "guardrails".
2.  ⚡ Working Memory – Kontekst bieżącej sesji (RAM). To, o czym rozmawiamy "teraz".
3.  🎯 Workflow Memory – *Nasz Secret Sauce*. Maszyna stanów (State Machine). Gdzie jestem w procesie biznesowym?
4.  📚 Episodic Memory – Co się wydarzyło? Historia interakcji, logi zdarzeń.
5.  🧠 Semantic Memory – Co wiem? Baza wiedzy (RAG), fakty o świecie i użytkowniku.
6.  🛠️ Procedural Memory – Jak to zrobić? Definicje narzędzi, API, instrukcje obsługi.
7.  **🔧 Resource Memory** –  ✅ Czy mam zasoby? Stan systemu, dostępność API, limity.


![](../../module2/images/memory_blue_print.jpg)


Mamy trzy krytyczne punkty, o które musisz zadbać w kontekście narzędzia (działania):

1.  **Discovery (odkrywanie)**. To jest ten moment selekcji. Wśród mnogości dostępnych serwerów, API i funkcji, musimy znaleźć to jedno właściwe – lub podjąć decyzję, że **żadne** nie pasuje. I tutaj zatrzymajmy się na moment, bo **sposób**, w jaki to zrobimy, jest krytyczny. Popularne podejście "wrzućmy wszystko do kontekstu i niech agent sobie wybierze" – jak już ustaliliśmy – totalnie nas nie interesuje. To droga donikąd. Zatem: jak inaczej? Odpowiedź może Cię zaskoczyć, bo wymaga pokory. Musimy zaakceptować fakt, że na obecnym poziomie technologii pewnych rzeczy nie osiągniemy "automagicznie". Musimy świadomie obniżyć poziom autonomii i zwiększyć poziom manualnego projektowania. Większość projektów "płynie" właśnie na fazie *Discovery*. Dlaczego? Bo po cichu zakładają możliwości (Poziom 6/7), które fizycznie jeszcze nie istnieją. My zrobimy to inaczej. Zamiast liczyć na cud, przejmiemy kontrolę i zaprojektujemy ten wybór "ręcznie", narzucając sztywne ramy. To jedyna opcja, by system działał stabilnie, a nie tylko efektownie wyglądał na demo.

2.  **Execution (wykonanie)**. Gdy już wiemy, co chcemy zrobić (mamy młotek), musimy to zrobić dobrze. Chodzi o dobranie właściwych argumentów i – co kluczowe – zarządzanie wynikiem. Często **nie chcemy** wrzucać całego wyniku (np. surowego HTML-a ze strony) z powrotem do kontekstu. Musimy tym jawnie sterować.

3.  **Security (bezpieczeństwo)**. Ten punkt niby stoi z boku, ale bez niego cała reszta jest tylko zabawką. Jeśli udostępniamy agenta na zewnątrz, integracja musi być bezpieczna. Bez filtrów i kontroli uprawnień nie wyjdziesz poza fazę demo.

W tej lekcji, w ramach naszej "drugiej iteracji", skupimy się mocno na **Discovery**. Zapomnijmy na chwilę o skomplikowanych protokołach (jak MCP) – potraktujmy je jako wizję "jak". My skupimy się na "co".

## Przegląd pola walki: Skąd wzięły się te strategie?

Zanim przejdziemy do pisania naszego kodu, zróbmy mały krok w bok i popatrzmy na kontekst rynkowy. To ważne, żebyś zrozumiał genezę problemu.

Twórcy najpopularniejszych platform (takich jak **Cursor** czy **Claude Code**) bardzo szybko zderzyli się ze ścianą. Zrozumieli brutalną prawdę: **ilość dostępnych narzędzi drastycznie degraduje jakość pracy modelu.** Im więcej opcji w kontekście, tym głupszy staje się agent.

Jaka była ich reakcja? Wprowadzili **sztywne limity**.
Systemy te często mówią wprost: "Maksymalnie możesz mieć podpięte X narzędzi" (często jest to liczba rzędu 50-80, czasem do 100).

![](../images/cursor_too_many.jpg)
    
I tu pojawia się problem inżynierski: jeden "tłusty" serwer MCP (np. pełna integracja z GitHubem czy AWS) potrafi wygenerować **więcej funkcji niż wynosi ten limit**.

To postawiło użytkowników pod ścianą. Skoro nie mogę mieć wszystkiego włączonego na raz ("plug & play" przestało działać), muszę zacząć tym zarządzać. Ta konieczność matką wynalazku – wymusiła powstanie strategii radzenia sobie z nadmiarem, które teraz omówimy.

Cztery główne podejścia, które wyłoniły się w boju:

### 1. Strategia "Manual Toggle" (ręczne wyłączanie)


To podejście typu "brute force", które możesz spotkać w narzędziach deweloperskich (np. Cursor). Skoro model głupieje przy 50-80 narzędziach, narzucamy sztywny limit.

System mówi wprost: "Możesz mieć maksymalnie 20 aktywnych narzędzi". Ty, jako człowiek, musisz wejść w ustawienia i "odptaszkować" to, co niepotrzebne.

![](../images/manual_disable_tools.jpg)

To działa doraźnie ("Quick updates"), ale jest bolesne w codziennej pracy. Wymaga od Ciebie ciągłego przełączania kontekstu. Musisz myśleć za agenta, zanim jeszcze zadasz pytanie. To pokazuje jednak ważną rzecz: **selekcja jest niezbędna**.

### 2. Strategia "Toolkits" (zestawy tematyczne)

To ewolucja poprzedniego pomysłu. Zamiast wybierać pojedyncze narzędzia, grupujemy je w paczki – analogicznie do tego, jak robimy to w repozytoriach na GitHubie. Tworzymy gotowe zestawy ("Front-end Kit", "Accounting Kit", "AWS Kit"). Agent dostaje dostęp nie do "wszystkiego", ale do konkretnej skrzynki z narzędziami dopasowanej do jego roli.
![](../images/mcp_toolkits.jpg)

To świetne rozwiązanie dla powtarzalnych, specyficznych zadań. Problem pojawia się w momentach "granicznych". Co jeśli "Księgowy" nagle potrzebuje jednego narzędzia z zestawu "Programisty"? System staje się sztywny. Niemniej, koncepcja **grupowania domendowego** jest bardzo wartościowa i będziemy z niej czerpać.



### 3. Strategia "Search & Call"

To podejście jest dla nas kluczowe, ponieważ idealnie mapuje się na filary, o których uczymy. Zamiast dawać agentowi dostęp do całej "skrzynki z narzędziami" (i zaśmiecać kontekst), dajesz mu tylko dwa mechanizmy:

1.  **`search_tool(query)`** – To jest czyste **Discovery**. Agent zgłasza potrzebę ("chcę wystawić fakturę"), a system zwraca mu definicję odpowiedniego narzędzia.
2.  **`call_tool(name, args)`** – To jest czyste **Execution**. Gdy agent już wie, co i jak wywołać, zleca to wykonanie.

**Dlaczego to jest ważne?**
Idea posiadania "wyszukiwarki narzędzi" (Discovery) brzmi w teorii genialnie, bo rozwiązuje problem limitów kontekstu. Jednak jak to zrobić na poziomie implementacji?

"Wyszukiwarkę" można zbudować na różne sposoby, co determinuje poziom zaawansowania systemu:
- Może to być **Keyword Search** (proste dopasowanie słów)
- Może to być **Semantic Search** (baza wektorowa) przeszukująca dokumentację
- Może to być **Hybrid Search** - połączenie wyszukiwarki klasycznej oraz wektorowej
- Może to być wreszcie **LLM** (Router), który swoim "rozumem" decyduje, co wybrać z ukrytej listy

![](../images/mcp_search_call.jpg)

My w tej lekcji mocno zainspirujemy się tym podejściem. Chcemy mieć mechanizm, który **odkrywa** właściwe narzędzie na żądanie, zamiast trzymać je wszystkie w pamięci operacyjnej. Niezależnie od tego, czy pod spodem użyjemy wektorów, czy Structured Output – kierunek jest ten sam: **Discovery musi być oddzielnym krokiem przed Execution.**




### 4. Strategia "Dynamic Tools" (żywe środowisko)

To jest wizja, do której dąży np. protokół MCP (Dynamic MCP), ale my spojrzymy na nią szerzej, architektonicznie. Słowo klucz to **Dynamic**. Zakładamy, że lista narzędzi nie jest wyryta w kamieniu na początku rozmowy. Ona żyje własnym, równoległym życiem. 

![](../images/mcp_dynamic_tools.jpg)

Możemy sobie wyobrazić, że niezależnie od tego, co robi agent, w tle działają procesy, które dodają lub odejmują dostępne narzędzia.
- Logujesz się jako Admin? *Pyk* – pojawiają się narzędzia do zarządzania bazą.
- Wchodzisz do folderu z projektem finansowym? *Pyk* – ładuje się "Toolkit Księgowy", a znika "Toolkit Marketingowy".

Tutaj pojawia się problem synchronizacji stanu. Jeśli agent na początku rozmowy wczyta sobie listę narzędzi do kontekstu (snapshot), a minutę później ta lista ulegnie zmianie w systemie – agent będzie działał na nieaktualnych danych (*stale data*). To klasyczny problem w systemach rozproszonych.

Co my z tym zrobimy? No właśnie... I tutaj dochodzimy do sedna. Zmapujemy ten problem "dynamiczności" na warstwę **Resource Memory** (Pamięć Zasobów), o której już mówiliśmy w Module 2 (a w tym module możemy dotknąć je). Traktujemy *Resource Memory* jako **kontener stanu**, który trzyma "aktualną prawdę" o dostępnych narzędziach. Agent, wykonując fazę **Discovery**, zawsze odpytuje ten kontener o bieżący stan. Dzięki temu nie martwimy się o "wpychanie" zmian do kontekstu na siłę – po prostu przy każdej decyzji agent zagląda do szuflady (pamięci), w której ktoś (system) mógł w międzyczasie podmienić zawartość.

To podejście daje nam ogromną elastyczność bez konieczności skomplikowanej orkiestracji powiadomień. Lista narzędzi żyje, a agent po prostu do niej zagląda wtedy, kiedy potrzebuje.

## Wnioski i plan gry

Przeglądając te różne strategie, jeden wniosek nasuwa się sam: nie ma idealnej, w pełni automatycznej metody zarządzania wyborem narzędzi. Przynajmniej jeszcze nie teraz.

Jednak najważniejsza lekcja, jaką musisz z tego wyciągnąć, brzmi: tym wyborem **TRZEBA** zarządzać. Pozostawienie tego przypadkowi (lub "domyślności" modelu) to prosty przepis na porażkę. Pytanie nie brzmi "czy", ale "jak".

Skoro wiemy, że pełna autonomia (Poziom 6/7), gdzie system "sam magicznie wie", jest jeszcze poza zasięgiem technologicznym, musimy zejść na ziemię. Pomyślmy o tym. Na poziomie 4 oznacza to wprowadzenie elementów inżynierii ludzkiej. Musimy ręcznie zaprojektować ramy, w których porusza się model.

###  Synteza najlepszych pomysłów

W tym module zastosujemy podejście, które rodzi się na styku rynkowych doświadczeń i naszej architektury pamięci. Stworzymy rozwiązanie hybrydowe, które łączy elastyczność AI (Software 3.0) ze stabilnością kodu (Software 1.0).

Brzmi ciekwie? No to zobacz jak to zrobimy:

1.  Fundament: `Resource Memory`. Jak pewnie się domyślasz, potrzebujemy jednego punktu prawdy. Resource Memory będzie naszym "rejestrem", który przechowuje informacje o tym, co jest dostępne i jakie ma metadane.
- W tej lekcji dla uproszczenia będzie to np. struktura JSON lub słownik w kodzie
- W produkcji może to być baza danych lub dynamiczny serwis
- Kluczowe jest to, że agent ma punkt synchronizacji, do którego może się odwołać

2.  Filtracja i dynamiczne pola. Nie chcemy, aby LLM wybierał spośród *wszystkiego*. Chcemy, aby wybierał spośród tego, co *ma sens* w danym momencie. Użyjemy do tego **Structured Output**. Genialne jest to, że pola w strukturze (np. lista dostępnych narzędzi w Literal) możemy wypełniać dynamicznie, zaciągając je wprost z naszej Resource Memory.

3.  Software 3.0 + Software 1.0 (stabilizacja przez kod). To jest kluczowy moment architektury. Musimy wyciągnąć wnioski z błędów poprzednich podejść:
- W podejściu naiwnym wrzucaliśmy do kontekstu surowe wyniki API, co "zatruwało" pamięć
- W *Code Mode* (generowanie kodu przez LLM) dostajemy "czarną skrzynkę" – nigdy nie mamy 100% pewności, co model tam wymyśli i czy się nie wyłoży. Dlatego my budujemy "bezpieczny kontener". Zasada jest prosta: **Software 3.0 (LLM)** służy do intencji i decyzji, a **Software 1.0 (Twój kod)** do wykonania. Oznacza to, że pewne klocki, które na Poziomie 4 są dla modelu "zbyt kruche", przenosimy do klasycznego programowania.

Przykład: Jeśli agent ma sprawdzić pogodę dla 100 miejscowości
- Rola LLM: Wygenerowanie listy tych miejscowości (np. JSON)
- Rola kodu: Uruchomienie pętli `for` i odpytanie API dla każdego elementu

Nie każemy modelowi "wykonywać pętli" ani "gadać z API" wewnątrz swojej wyobraźni. Robimy to jawnie w kodzie.

#### Czy to wprowadza sztywność?
Tak. 

I robimy to w pełni świadomie. To jest "wybór bez wyboru". Skoro LLM na tym poziomie (poniżej 5) jesteśmy, to nie ma jak dostarczyć powtarzalnej jakości przy skomplikowanej logice wykonawczej, musimy (my, inżynierowie) wziąć to na siebie. Odzyskujemy kontrolę i stabilność kosztem elastyczności. Choć dobra wiadomość jest taka, że te "kawałki kodu" (np. obsługę pętli czy requestów) piszemy raz i są one w pełni re-używalne.

Dzięki temu budujemy system, który jest wystarczająco inteligentny, by być elastycznym, ale wystarczająco ograniczony, by być bezpiecznym.

## Nowy Use Case: Research Assistant

Skoro mamy już teorię, czas na praktykę. Najlepiej uczyć się na żywym organizmie, dlatego rozpoczynamy pracę nad nowym przypadkiem użycia: **Asystentem Badawczym**.

Trzeba sobie od razu powiedzieć wprost: ten temat docelowo zasługuje na **Poziom 5**, ponieważ skuteczny research wymaga zaawansowanego planowania, stawiania hipotez i ich weryfikacji. Będziemy go tam rozwijać w kolejnym module.

Jednak na razie celowo zrealizujemy go jako solidną wersję na **Poziomie 4**.

Dlaczego? Bo musimy najpierw utrwalić fundamenty **Discovery & Execution**. Dopiero gdy poczujesz się pewnie z mechanizmami doboru narzędzi na tym etapie, będziemy gotowi, by wejść wyżej. Ale uwaga – już teraz, nawet w tej uproszczonej wersji, to narzędzie ma ogromną wartość biznesową, bo pokazuje różnicę między "chatbotem, który zmyśla", a "agentem, który sprawdza".


## Mapowanie domeny

W poprzednich lekcjach (przy okazji Restauracji, Taxi i Tutora) nauczyliśmy się ważnej zasady: nie siadamy do kodu, dopóki nie zrozumiemy "fizyki" problemu. Pamiętasz o tym?

To, że nazwiemy nasz system "Asystentem Badawczym", dla każdego może znaczyć coś innego.
- Dla marzyciela: System, który przeczyta 1000 książek i napisze doktorat.
- Dla inżyniera: System, który w skończonym czasie i budżecie znajdzie wiarygodną odpowiedź na konkretne pytanie.

Musimy zmapować nasze marzenia na dostępne **narzędzia**. Zobaczmy, co mamy w "garażu" (nasze API), aby wiedzieć, jaki dom możemy z tego zbudować.


### Definicja Sukcesu (*Definition of Done*)

Zdefiniujmy, czym jest nasz *Research Assistant* w wersji MVP (Poziom 4).

Nie budujemy "przeglądarki internetu". Budujemy "inteligentnego skauta Tematycznego".

Jego zadaniem jest:
1.  Przyjąć pytanie o konkretny, świeży temat (np. "Co nowego w Python 3.13?")
2.  Przeszukać źródła (możemy stworzyć zaufaną listę)
3.  Na podstawie fragmentów (`raw_content` z wyszukiwania) ocenić, który **jeden lub dwa** artykuły są warte pełnego przeczytania
4.  Pobrać ich treść i wygenerować syntetyczną odpowiedź z cytowaniem źródła

To jest podejście inżynierskie: **Jakość > Ilość**. Zamiast przepalać dolary na czytanie śmieciowych stron SEO, robimy chirurgiczne cięcie.

### Scenariusze (DDD Lite)

Zmapujmy to teraz na konkretne sytuacje, tak jak robiliśmy to przy Restauracji itd. To pozwoli nam zaplanować logikę **Discovery**.

#### Scenariusz 1: Happy Path (Idealny strzał)
*   **User:** "Sprawdź na Hacker News, co ludzie mówią o MCP."
*   **Agent (Discovery):** Wybiera narzędzie `search` z filtrem `domain="news.ycombinator.com"`.
*   **Wynik:** Dostaje 3 wyniki. Widzi, że pierwszy ma wysoki `score` i tytuł "MCP explained without hype".
*   **Agent (Decision):** "Ten jeden link wygląda obiecująco. Pobieram go (`extract`)."
*   **Agent (Final):** Generuje podsumowanie na podstawie pobranej treści.

#### Scenariusz 2: Szum informacyjny (Konieczność filtracji)
*   **User:** "Co nowego w AI?" (Pytanie bardzo ogólne).
*   **Agent (Discovery):** Robi szeroki `search`.
*   **Wynik:** Dostaje 5 linków: 3 to "gadu-gadu", 1 to stary artykuł z 2021, 1 to news z wczoraj.
*   **Wyzwanie:** Tutaj trzeba zadziałać inteligentnie. Agent analizuje metadane (datę, fragment treści). Decyduje: "Linki 1-4 są bezwartościowe. Pobieram tylko link nr 5".
*   **Wniosek:** To oszczędza nam wywołania `extract` na śmieciowe strony.

#### Scenariusz 3: Ślepy zaułek (Zero results)
*   **User:** "Jaka jest pogoda w Narnii?"
*   **Agent (Discovery):** Robi `search`. Wyniki są puste lub nonsensowne.
*   **Decyzja:** Nie próbuje na siłę robić `extract`. Od razu zwraca `final_answer`: "W moich zaufanych źródłach brak informacji na ten temat".
*   **Wartość:** Unikamy halucynacji. Agent na Poziomie 4 wie, kiedy nie wie.

### Dlaczego to jest ważne?

W Module 5 będziemy wprowadzać **Leaderboardy** i **Quality Checks**. Będziemy tam formalnie mierzyć jakość agenta za pomocą binarnej ewaluacji (np. "Czy odpowiedź zawiera link źródłowy?", "Czy odpowiedź pochodzi z zadanego roku?").

Już teraz, projektując ten system, musimy o tym myśleć. Nasz Asystent nie ma być "gadugadu". Ma być precyzyjnym narzędziem, które dostarcza dowody (evidence).


Przejdźmy do implementacji.

## Wczytujemy biblioteki

In [ ]:
from pydantic import BaseModel, Field, create_model
from typing import Literal, Union
from enum import Enum
from typing import Literal
import helper as h

## Implementacja warstwy *discovery*

Mamy już plan działania ("Mapowanie domeny"). Wiemy, że użytkownik może zapytać o wszystko, ale my nie chcemy, aby Agent szukał "wszędzie".

W Software 3.0, zanim napiszemy logikę (prompt), musimy zdefiniować strukturę danych. To jest nasz inżynierski sposób na okiełznanie rzeczywistości.

Zamiast jednej, wielkiej funkcji `search(query)`, która jest workiem bez dna, przyjmijmy **trzy pragmatyczne scenariusze**. To jest nasze "zauważenie" – założenie projektowe, które porządkuje intencje użytkownika na trzy konkretne kubełki.

Dlaczego akurat te trzy?

### 1. `WebSearch` (siatka bezpieczeństwa)
To jest nasze narzędzie "ogólnego przeznaczenia". Jeśli intencja użytkownika nie pasuje do żadnej specyficznej kategorii, wpada tutaj.
- **Kiedy używamy:** Definicje, tutoriale, ogólne fakty (`"Czy koty w Krakowie śpią dłużej niż w Warszawie?"`).
- **Charakterystyka:** Szukamy szeroko. To jest nasz fallback.

### 2. `NewsSearch` (wymiar czasu)
Tutaj kluczowa jest **świeżość**. Zwykła wyszukiwarka często zwraca "najpopularniejsze" wyniki (SEO), które mogą być stare. Narzędzie typu NewsSearch działa inaczej – priorytetyzuje to, co stało się wczoraj lub dzisiaj.
- Kiedy używamy: `"Ostatnie zmiany w podatkach"`.
- Charakterystyka: Wykrywamy intencję czasową. Jeśli w pytaniu czuć "teraz" lub "ostatnio", to jest właściwy kubełek.

### 3. `DomainSearch` (zaufane źródła)
To jest nasza "snajperka". Najciekawszy element tej układanki.
Często nie chcemy przeszukiwać "całego Internetu" (bo jest tam dużo szumu i marketingu). Chcemy sprawdzić, co mówi konkretna, hermetyczna społeczność lub zaufane źródło.
- Kiedy używamy: "Co ludzie na Hacker News sądzą o modelu o1?", "Najnowsze prace naukowe (Arxiv) o pamięci AI".
- Charakterystyka: Zamiast szukać poziomo (wszystko), szukamy pionowo (tylko w zaufanym silosie). Dzięki temu, że z góry projektujemy listę tych "zaufanych domen" (np. `hackernews`, `academic`, `docs`), drastycznie podnosimy jakość (*Signal-to-Noise Ratio*).


### ☝️ Zwróć uwagę:
Traktuj ten podział jako punkt wyjścia. Ta lista nie jest zamknięta. W Twoim projekcie możesz dodać `StockSearch` (giełda) albo `InternalDocSearch` (Twoja baza Notion). Chodzi o to, aby zobaczyć, że Ty jako inżynier narzucasz strukturę, w której porusza się model.


## 🧠 Zadanie 3.4.1: Zdefiniuj WebSearchTool (Siatka bezpieczeństwa)

Twoim pierwszym zadaniem jest zdefiniowanie modelu Pydantic dla najbardziej ogólnego narzędzia: `WebSearchTool`. To będzie nasz "fallback" – jeśli zapytanie użytkownika nie pasuje do żadnej specyficznej kategorii, trafi właśnie tutaj.

### Wymagania:
1.  Stwórz klasę `WebSearchTool` dziedziczącą po `BaseModel`.
2.  Zdefiniuj pole `tool_name` jako `Literal["web_search"]` (to będzie nasz identyfikator dla routera).
3.  Zdefiniuj pole `query` typu `str`. **Kluczowe:** Dodaj opis (`description`), który poinstruuje model, że zapytanie ma być "znormalizowane i precyzyjne" (np. zamiast "ej, weź mi znajdź...", ma być "definicja pętli while python").
4.  Zdefiniuj pole `max_results` typu `int` z wartością domyślną `3`. Ogranicz je (walidacja Pydantic `ge=1`, `le=5`), aby agent nie przesadzał z ilością danych.

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;"> 👉 Kliknij tutaj (1 klik), aby zobaczyć podpowiedź 👈 </summary>
<p>

### Podpowiedź:
1.  Nie zapomnij zaimportować `BaseModel`, `Field` oraz `Literal` z biblioteki `pydantic` i `typing`.
2.  W polu `query`, opis (`description`) jest najważniejszy. To instrukcja dla LLM-a. Napisz tam wprost: "Znormalizowane zapytanie do wyszukiwarki".
3.  Do ograniczenia liczb użyj parametrów `ge` (greater or equal) i `le` (less or equal) wewnątrz funkcji `Field()`.

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;"> 👉 Kliknij tutaj (1 klik), aby zobaczyć odpowiedź 👈 </summary>
<p>

```python

class WebSearchTool(BaseModel):
    """
    Narzędzie do ogólnego wyszukiwania w Internecie.
    Używane, gdy intencja użytkownika nie pasuje do newsów ani specyficznych domen.
    """
    tool_name: Literal["web_search"] = "web_search"
    query: str = Field(
        ..., 
        description="Znormalizowane, precyzyjne zapytanie do wyszukiwarki (bez zbędnych słów grzecznościowych)."
    )
    max_results: int = Field(
        default=3, 
        ge=1, 
        le=5, 
        description="Liczba wyników do pobrania."
    )
```

</p>
</details>
</p>
</details>

In [ ]:
## YOUR CODE HERE

## 🧠 Zadanie 3.4.2: Zdefiniuj `NewsSearchTool` 

Teraz dodajmy wymiar czasu. Zwykła wyszukiwarka często zwraca stare, dobrze wypozycjonowane artykuły. My chcemy narzędzia, które priorytetyzuje świeżość.

### Wymagania:
1.  Stwórz klasę `NewsSearchTool`.
2.  Nazwa narzędzia (`tool_name`) to `news_search`.
3.  Oprócz standardowego `query` i `max_results`, dodaj nowe, kluczowe pole: `time_range`.
4.  Pole `time_range` musi być ograniczone (Literal) tylko do dwóch wartości: `"day"` (ostatnie 24h) oraz `"week"` (ostatni tydzień). Ustaw wartość domyślną na `"day"`.

### Dlaczego to ważne?
Dając modelowi wybór tylko między `day` a `week`, zmuszasz go do kategoryzacji. Gdybyś dał mu pole tekstowe `date`, mógłby zacząć halucynować formaty dat, których Twoje API nie obsługuje.

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;"> 👉 Kliknij tutaj (1 klik), aby zobaczyć podpowiedź 👈 </summary>
<p>

### Podpowiedź:
1.  Skopiuj strukturę z poprzedniego zadania, to przyspieszy pracę.
2.  W polu `time_range` użyj `Literal["day", "week"]`. To zadziała jak enum w bazie danych – model nie będzie mógł wstawić tam nic innego (np. "month" wyrzuci błąd walidacji, co jest pożądane).
3.  Dodaj opis do `time_range`, np. "Zakres czasu dla wyszukiwanych newsów".

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;"> 👉 Kliknij tutaj (1 klik), aby zobaczyć odpowiedź 👈 </summary>
<p>

```python
class NewsSearchTool(BaseModel):
    """
    Wyszukiwanie newsów i bieżących wydarzeń.
    Priorytet ma świeżość informacji (ostatnie dni/tygodnie).
    """
    tool_name: Literal["news_search"] = "news_search"
    query: str = Field(..., description="Temat newsa lub wydarzenia.")
    time_range: Literal["day", "week"] = Field(
        default="day", 
        description="Zakres czasu: 'day' dla najświeższych newsów, 'week' dla przeglądu tygodnia."
    )
    max_results: int = Field(default=3, ge=1, le=5)
```

</p>
</details>
</p>
</details>


In [ ]:
## YOUR CODE HERE

## 🧠 Zadanie 3.4.3: Zdefiniuj `DomainSearchTool` (zaufane źródła)

To jest najważniejsze zadanie. Stworzysz narzędzie-"snajpera", które szuka tylko w zaufanych źródłach. To drastycznie podnosi jakość wyników.

### Wymagania:
1.  Stwórz klasę `DomainSearchTool`.
2.  Nazwa narzędzia (`tool_name`) to `domain_search`.
3.  Zdefiniuj pole `domain_preset`. To nie może być dowolny string (bo model mógłby wpisać "google.com"). Użyj `Literal` z trzema konkretnymi wartościami:
    - `hackernews` – dla opinii społeczności technologicznej
    - `academic` – dla wyszukiwania papierów naukowych
    - `python` – dla oficjalnej dokumentacji Pythona
4.  W opisie pola (`description`) wyjaśnij modelowi, kiedy używać którego presetu (np. "hackernews do opinii, academic do badań"). To jest Twoja "logika biznesowa" zaszyta w definicji typu!

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;"> 👉 Kliknij tutaj (1 klik), aby zobaczyć podpowiedź 👈 </summary>
<p>

### Podpowiedź:
1.  To pole `domain_preset` jest sercem tego narzędzia. Dzięki niemu, zamiast szukać "opinie o AI" w całym internecie (gdzie jest dużo spamu), model wymusi szukanie w domenie `ycombinator.com` (jeśli tak to obsłużysz w kodzie execution).
2.  Opis pola jest krytyczny. Jeśli nie napiszesz, czym jest `hackernews`, model może tego nie skojarzyć z "opiniami ekspertów". Bądź precyzyjny.

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;"> 👉 Kliknij tutaj (1 klik), aby zobaczyć odpowiedź 👈 </summary>
<p>

```python
class DomainSearchTool(BaseModel):
    """
    Wyszukiwanie precyzyjne w konkretnej, zaufanej domenie lub społeczności.
    Używane, gdy zależy nam na wysokiej jakości źródła (opinie ekspertów, badania).
    """
    tool_name: Literal["domain_search"] = "domain_search"
    query: str = Field(..., description="Specyficzne zagadnienie do wyszukania.")
    domain_preset: Literal["hackernews", "academic", "python"] = Field(
        ...,
        description=(
            "Wybierz preset: "
            "'hackernews' - dyskusje, opinie społeczności tech; "
            "'academic' - publikacje naukowe, papers; "
            "'python' - oficjalna dokumentacja i PEP."
        )
    )
    max_results: int = Field(default=3, ge=1, le=5)
```

</p>
</details>
</p>
</details>

In [ ]:
## YOUR CODE HERE

Mamy już gotowe cegiełki (Twoje trzy klasy narzędzi). Teraz czas na cement, który połączy to wszystko w jeden inteligentny mechanizm decyzyjny.

Wróćmy do naszej głównej myśli: Discovery to moment wyboru.

Chcemy, aby nasz Agent spojrzał na pytanie użytkownika, pomyślał chwilę, a następnie wybrał jedno, najlepsze narzędzie z dostępnego zestawu.

Jak wyrazić to w kodzie? Jak powiedzieć modelowi: "W tym polu możesz wstawić obiekt `typu A` lub `typu B` lub `typu C`"?

Z pomocą przychodzi nam `Union`.

W Pythonie (i w Pydantic) `Union[X, Y, Z]` oznacza dokładnie to: "To pole może przyjąć wartość, która pasuje do schematu X, Y lub Z". Dla modelu LLM jest to sygnał, że stoi przed wyborem wielokrotnego wyboru (multiple choice), ale może zaznaczyć tylko jedną odpowiedź.


To jest doskonały moment na "stop klatkę". Musimy wyjaśnić ten pozorny paradoks, zanim pójdziesz dalej.

Czujny obserwator – czyli Ty – może teraz podnieść rękę i zapytać: 

> *"Zaraz, zaraz! Przecież w poprzednich lekcjach krytykowaliśmy podejście, w którym wrzucamy definicje narzędzi do kontekstu i liczymy na cud. A teraz robimy dokładnie to samo, tylko używając Pydantica. O co chodzi?"*

Masz rację, mechanizm jest podobny (model musi "widzieć" narzędzia), ale **sposób wykonania** zmienia tutaj wszystko. Różnica między naiwnym wrzuceniem listy z MCP a naszym podejściem jest jak różnica między "szukaniem w google" a "szukaniem w encyklopedii".

Dlaczego to rozwiązanie jest stabilniejsze:

1.  Chirurgiczna precyzja (`Schema` vs `Text`). W naiwnym podejściu często dostajesz automatycznie wygenerowane opisy z kodu (np. z docstringów), które bywają niejasne i ... wprowadzające w błąd. Tutaj, używając Pydantic (`Field` + `description`), ręcznie zaprojektowaliśmy każdą wskazówkę. Model nie zgaduje – on wypełnia formularz.

2.  Wymuszony wybór (`Union`). To jest klucz. Wrzucając narzędzia "luzem" do promptu, model może próbować łączyć je w dziwne kombinacje albo wymyślać własne parametry. Używając `Union[...]`, stawiamy sprawę twardo: Musisz wybrać dokładnie jeden obiekt z tej listy. Nie możesz wybrać "trochę tego, trochę tamtego". To działa jak fizyczna zwrotnica na torach.

3.  Kwestia skali (zarządzanie ilością). I tu dochodzimy do sedna. Oczywiście, gdy wrzucisz do tego `Union` 150 narzędzi, system mógłby się wyłożyć (tak samo jak w naiwnym podejściu). Chociaż nawet w tym przypadku nadal może być dość stabilny… bo jest struktura.
    
Dlatego architektura wygląda tak:
- Na poziomie *Resource Memory* (o którym mówiliśmy wcześniej) filtrujesz do mniejszej ilości narzędzi do np. 3-5 najbardziej prawdopodobnych. Dopiero te 3-5 (może być więcej, ale bez przesady) narzędzi wrzucasz do `Union`.

Na tym etapie (Poziom 4) możemy drastycznie zwiększyć jakość tego wyboru, pokazując mu mały, precyzyjny wycinek rzeczywistości i narzucając sztywne typy danych.

Zdefiniujmy więc nasz główny plan działania, czyli klasę `ResearchPlan`.

In [ ]:
class ResearchPlan(BaseModel):
    """
    Reprezentuje proces decyzyjny agenta Discovery.
    
    Składa się z:
    - procesu myślowego (thought_process),
    - wyboru konkretnego narzędzia (tool) na podstawie tego procesu.
    """

    thought_process: str = Field(
        ...,
        description=(
            "Wyjaśnienie (Chain of Thought), dlaczego wybrano to konkretne narzędzie. "
            "Model musi najpierw uzasadnić wybór, zanim określi wartość pola `tool`."
        ),
    )

    # Mechanizm Discovery:
    # Union ogranicza model do wyboru *jednego* z dostępnych narzędzi.
    # Dzięki temu agent nie może wymyślić własnej opcji – musi wybrać najlepsze dopasowanie.
    tool: Union[WebSearchTool, NewsSearchTool, DomainSearchTool] = Field(
        ...,
        description="Narzędzie wybrane jako najbardziej pasujące do intencji użytkownika."
    )


### Jak to czytać?

Konstrukcja `Union[A, B, C]` to dla modelu instrukcja: *"Stoisz przed wyborem wielokrotnym (multiple choice). Przeanalizuj swoje zadanie, przeczytaj definicje A, B oraz C, a następnie zwróć mi obiekt, który pasuje do **jednego** z nich."*

Dzięki temu, jeśli użytkownik zapyta o "opinie na temat nowego frameworka", model:
1.  Przeanalizuje opisy (description).
2.  Zrozumie, że `NewsSearch` to nie to, a `WebSearch` jest za ogólny.
3.  Wybierze `DomainSearchTool` (bo tam zdefiniowałeś "opinie społeczności"), precyzyjnie wypełniając jego pola.


Zwróć uwagę na pole `thought_process`. Zmuszamy model do "myślenia" przed działaniem. To może pomóc zwiększyć precyzję wyboru. Agent musi najpierw „napisać sobie”, dlaczego uważa, że to jest news, a nie artykuł naukowy, a dopiero potem wypełnić pole `tool`.


Jeśli użytkownik zapyta o `"opinię o nowym modelu AI"`, model przeanalizuje opisy i zrozumie:
- `WebSearch`? Za ogólne.
- `NewsSearch`? Może, ale pytanie jest o opinie.
- `DomainSearch` (z presetem `hackernews`)? Idealne, bo w opisie jest "opinie społeczności".

Więc w pole `tool` wstawi obiekt `DomainSearchTool`.

To jest właśnie **Discovery na Poziomie 4**. Mimo że lista narzędzi jest skończona i zdefiniowana przez nas, wybór odbywa się dynamicznie na podstawie zrozumienia intencji przez LLM.

Oto ostatni element układanki. Mamy zdefiniowane narzędzia (klasy Pydantic) oraz strukturę decyzyjną (`ResearchPlan` z `Union`). Teraz musimy napisać instrukcję ("biznesową logikę"), która powie modelowi, jak tych narzędzi używać, i wykonać faktyczne wywołanie.

## 🧠 Zadanie 3.4.4: Implementacja Routera i System Prompt

To jest moment, w którym łączymy wszystko w całość. Twoim zadaniem jest napisanie **System Promptu**, który będzie pełnił rolę instrukcji dla modułu Discovery, oraz wykonanie zapytania do API.

### Wymagania:

1.  Zdefiniuj zmienną `SYSTEM_PROMPT`. Prompt musi realizować 3 kroki:
    *   Zrozumieć intencję.
    *   Znormalizować zapytanie (np. usunąć "Cześć, poszukaj mi..." i zostawić samo sedno).
    *   Wybrać narzędzie według zasad: "nowości" -> `news_search`, "opinie/HackerNews" -> `domain_search`, reszta -> `web_search`.
2.  Przygotuj listę `messages`.
3.  Wykonaj zapytanie, używając naszej funkcji pomocniczej `h.call_api` (lub klienta OpenAI), przekazując `ResearchPlan` jako oczekiwany format wyniku (`response_model`).
4.  Przetestuj działanie na dwóch różnych pytaniach (np. o Hacker News i o publikacje naukowe), aby zobaczyć, czy routing działa.

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;"> 👉 Kliknij tutaj (1 klik), aby zobaczyć podpowiedź 👈 </summary>
<p>

### Podpowiedź:
1.  W System Prompcie nie musisz opisywać technicznych szczegółów JSON-a (to robi Pydantic). Skup się na logice biznesowej. Napisz wprost mapowanie: "Jeśli użytkownik pyta o opinie -> użyj domain_search z presetem hackernews".
2.  Normalizacja query jest kluczowa. Poproś model, aby czyścił zapytanie ze zbędnych słów i dodawał kontekst (np. rok), jeśli to istotne.
3.  Pamiętaj, aby w kodzie podmieniać `user_question`, żeby przetestować różne ścieżki (routing).

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;"> 👉 Kliknij tutaj (1 klik), aby zobaczyć odpowiedź 👈 </summary>
<p>

```python

SYSTEM_PROMPT = """Jesteś modułem Discovery asystenta badawczego.

Twoje zadanie:
1. Zrozum intencję użytkownika
2. Znormalizuj zapytanie (precyzyjne, bez zbędnych słów)
3. Wybierz najlepsze narzędzie

Dostępne narzędzia:
- web_search: ogólne pytania, definicje, tutoriale
- news_search: bieżące wydarzenia, ogłoszenia (ostatnie dni)
- domain_search: opinie społeczności (hackernews), artykuły naukowe (academic), dokumentacja Python (python)

Zasady:
- Jeśli pytanie zawiera "co nowego", "ostatnio", "news" → news_search
- Jeśli pytanie o opinie, dyskusje, "co ludzie myślą" → domain_search (hackernews)
- Jeśli pytanie o badania, papers → domain_search (academic)
- W pozostałych przypadkach → web_search

Dodaj kontekst czasowy do query jeśli istotny (np. "Python 3.13 2024").
"""


user_question = "Sprawdź na Hacker News, co ludzie mówią o MCP"
# user_question = "najnowsze publikacji o pamięci dla agentów AI"

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": user_question}
]


# Zakładamy, że ResearchPlan został zdefiniowany w sekcji wyjaśniającej Union
result = h.call_api(messages, ResearchPlan)


print(f"User Query: {user_question}")
print("-" * 20)
print(result)
```

</p>
</details>
</p>
</details>

In [ ]:
## YOUR CODE HERE

## Zadanie 3.4.5**: *Resource Memory* i dynamiczne modele 

To zadanie ma "gwiazdkę", bo wchodzimy głęboko w metaprogramowanie. Do tej pory nasz `ResearchPlan` był statyczny – "wyryty w kamieniu" w kodzie. Zawsze miał te same 3 narzędzia w `Union`.

Ale w prawdziwym świecie Software 3.0 (Enterprise) dostępność narzędzi zmienia się dynamicznie:
1.  **Uprawnienia:** Junior nie widzi narzędzi administracyjnych.
2.  **Koszty:** W trybie oszczędnym wyłączamy drogie API.
3.  **Kontekst:** Jeśli rozmawiamy o kodowaniu, nie potrzebujemy narzędzi do rezerwacji stolika.

Musimy sprawić, by nasza struktura Pydantic **reagowała** na te zmiany. Nie możemy ręcznie pisać klasy `ResearchPlanForAdmin`, `ResearchPlanForUser`, `ResearchPlanForGuest`. Musimy generować te klasy w locie.

### Analiza architektury 

Zanim przejdziesz do zadania, zrozum, co tu się wydarzy. Mamy trzy kluczowe elementy:

**1. Tool Registry (Katalog)**
To jest nasza baza danych. W tym prostym kodzie to słownik (`dict`), ale w produkcji to może być baza SQL lub plik konfiguracyjny. Tu leżą definicje wszystkich możliwych narzędzi.

**2. Resource Memory (Filtr)**
To jest mózg operacji. Funkcja `resource_memory` decyduje, **co w danej chwili jest dostępne**.
*   W kodzie poniżej zobaczysz prosty warunek: `if "hacker" in user_intent`.
*   **Ważne:** Traktuj to jako symbol. W prawdziwym systemie w tym miejscu byłby bardziej zaawansowany sposób wyszukiwania informacji, sprawdzenie uprawnień użytkownika w Active Directory albo szybkie wywołanie do innego modelu („Czy do tego zadania potrzebuję Google?”).

**3. Dynamiczny Model (`create_model`)**
To jest najtrudniejsza część. Używamy funkcji `create_model` z biblioteki Pydantic. Pozwala ona stworzyć nową klasę **w trakcie działania programu** (runtime).
Dzięki temu nasz `Union` nie jest sztywny. Jeśli `resource_memory` zwróci tylko jedno narzędzie, nasz model będzie miał `Union` z jednym elementem. Jeśli zwróci pięć – `Union` będzie miał pięć. LLM fizycznie nie będzie w stanie wybrać narzędzia, którego mu nie podamy.

### Co trzeba zrobić?

Twoim zadaniem jest sklejenie tego dynamicznego potoku. Kod jest gotowy, ale musisz go uruchomić i przeanalizować, jak zmienia się zachowanie agenta w zależności od pytania.

**Wymagania:**
1.  Przeanalizuj i uruchom dostarczony kod implementujący `build_dynamic_research_plan`.
2.  Sprawdź działanie dla zapytania zawierającego słowo "Hacker" (powinno dynamicznie zawęzić narzędzia tylko do `DomainSearchTool`).
3.  Sprawdź działanie dla innego zapytania (powinno udostępnić pełen zestaw narzędzi).

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;"> 👉 Kliknij tutaj (1 klik), aby zobaczyć podpowiedź 👈 </summary>
<p>

### Co tu się dzieje pod maską?

1.  **`create_model`**: To funkcja fabryczna. Zamiast pisać `class X(BaseModel): ...`, wywołujemy funkcję, która zwraca nam klasę.
2.  **`tool_annotation`**: To zmienna, która dynamicznie buduje typ `Union`. Jeśli na liście mamy `[A, B]`, to tworzy `Union[A, B]`.
3.  **Prompt Injection w `description`**: Zauważ, że dynamicznie generujemy też opis pola `tool`. Doklejamy tam listę dostępnych narzędzi (`tool_descriptions`), żeby model wiedział, z czego może wybierać. To jest "prompt engineering wewnątrz definicji typu".

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;"> 👉 Kliknij tutaj (1 klik), aby zobaczyć odpowiedź 👈 </summary>
<p>

```python
from pydantic import create_model
from typing import List, Type

# 1. TOOL REGISTRY - Nasza "baza danych" z definicjami
TOOL_REGISTRY: dict[str, type[BaseModel]] = {
    "web_search": WebSearchTool,
    "news_search": NewsSearchTool,
    "domain_search": DomainSearchTool,
}

# 2. RESOURCE MEMORY - Logika filtracji
def resource_memory(user_intent: str) -> list[type[BaseModel]]:
    """
    Decyduje, które narzędzia są potrzebne w danym kontekście.
    UWAGA: W produkcji tutaj byłby Semantic Search, RAG lub User Permissions check.
    Tutaj symulujemy to prostym if-em.
    """
    user_intent = user_intent.lower()
    
    # Symulacja: Jeśli użytkownik pyta o "hacker", system wie, 
    # że inne narzędzia są zbędnym szumem i zwraca tylko jedno.
    if "hacker" in user_intent:
        return [
            TOOL_REGISTRY["domain_search"]
        ]
    
    # Domyślnie zwracamy pełen zestaw (fallback)
    return [
        TOOL_REGISTRY["web_search"],
        TOOL_REGISTRY["news_search"],
        TOOL_REGISTRY["domain_search"]
    ]

# 3. DYNAMIC MODEL BUILDER - Metaprogramowanie
def build_dynamic_research_plan(available_tools: list[type[BaseModel]]) -> type[BaseModel]:
    """
    Tworzy klasę ResearchPlan w locie, z Union dopasowanym TYLKO do dostępnych narzędzi.
    """
    
    if not available_tools:
        raise ValueError("Resource memory zwróciła pustą listę narzędzi!")
    
    # Dynamiczne tworzenie typu Union
    if len(available_tools) == 1:
        tool_annotation = available_tools[0]
    else:
        # Konwertujemy listę klas na krotkę, bo Union wymaga krotki
        tool_annotation = Union[tuple(available_tools)]
    
    # Dynamiczne generowanie opisu dla modelu (żeby LLM wiedział co ma do wyboru)
    tool_descriptions = []
    for tool_class in available_tools:
        # Pobieramy domyślną nazwę narzędzia z pola tool_name
        name = tool_class.model_fields.get("tool_name")
        if name and name.default:
            tool_descriptions.append(f"- {name.default}: {tool_class.__doc__ or 'brak opisu'}")
    
    # Magia Pydantic: tworzenie klasy w runtime
    DynamicResearchPlan = create_model(
        'ResearchPlan',
        thought_process=(
            str,
            Field(
                ...,
                description=(
                    "Chain of Thought: wyjaśnij krok po kroku, "
                    "dlaczego wybierasz to narzędzie. "
                    "Najpierw uzasadnij, potem wypełnij `tool`."
                )
            )
        ),
        tool=(
            tool_annotation, # Tu wstrzykujemy nasz dynamiczny Union
            Field(
                ...,
                description=f"Wybierz JEDNO z dostępnych narzędzi:\n" + "\n".join(tool_descriptions)
            )
        )
    )
    
    return DynamicResearchPlan    


# --- URUCHOMIENIE I TESTY ---

# Przypadek A: Zapytanie specyficzne ("Hacker News")
user_question = "Sprawdź na Hacker News, co ludzie mówią o MCP"

print(f"\n🧪 Scenariusz A: '{user_question}'")
# Krok 1: Resource Memory filtruje narzędzia
available_tools = resource_memory(user_question)
print(f"   🔍 Resource Memory wybrało: {[t.__name__ for t in available_tools]}")

# Krok 2: Budujemy model
ResearchPlanModel = build_dynamic_research_plan(available_tools)

# Krok 3: Wywołanie LLM
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": user_question}
]

# Zauważ: przekazujemy ResearchPlanModel, który przed chwilą powstał!
result = h.call_api(messages, ResearchPlanModel)
print(f"   🤖 Decyzja agenta: Narzędzie={result.tool.tool_name}")


# Przypadek B: Zapytanie ogólne
print("-" * 40)
user_question_b = "Jaka jest pogoda?"
print(f"\n🧪 Scenariusz B: '{user_question_b}'")

available_tools_b = resource_memory(user_question_b)
print(f"   🔍 Resource Memory wybrało: {[t.__name__ for t in available_tools_b]}")
# Tutaj powinniśmy widzieć 3 narzędzia, bo if "hacker" nie zadziałał
```

</p>
</details>
</p>
</details>

In [ ]:
## YOUR CODE HERE

## Podsumowanie

Na koniec chciałbym, żebyśmy na moment oderwali się od kodu i spojrzeli na to, co tu właściwie zbudowaliśmy. Ta lekcja była o *Discovery*, ale wnioski z niej płynące są znacznie szersze.

Zwróć uwagę na jedną rzecz: przez całą tę lekcję intensywnie pracowaliśmy nad mechanizmem wyboru narzędzi, wprowadziliśmy zalążki *Resource Memory*, a słowo "MCP" (Model Context Protocol) pojawiło się właściwie tylko w tle.

Dlaczego? To celowe.

**MCP to tylko protokół.** To kabel, standard przesyłu. On może być niezwykle pomocny w standaryzacji, ale sam z siebie nie rozwiąże problemu architektury (na poziomie 4 lub poniżej). To, czy połączysz narzędzia przez MCP, przez REST API, czy przez lokalne funkcje Pythona, jest wtórne.

Kluczowe jest to, co zbudowałeś "nad" tymi narzędziami:
1.  **Discovery Layer:** Logika, która decyduje "CO" wybrać, zanim zacznie "JAK" to robić.
2.  **Dynamic Tools:** Mechanizm, który zamiast liczyć na "magię" i nadzieję, że model sam się domyśli, daje mu precyzyjne ramy (typ `Union`, dynamiczne opisy).

W świecie Software 3.0 łatwo ulec pokusie: "Podepnę wszystko do wielkiego wora kontekstu i niech się dzieje wola nieba". My poszliśmy inżynierską drogą.

Dzięki temu, że masz teraz ten stabilny rdzeń (Router + Resource Memory), w przyszłości możesz podpiąć pod niego cokolwiek.
- Chcesz podpiąć serwer MCP z 50 narzędziami? Proszę bardzo – Twój router je przefiltruje
- Chcesz podpiąć stare API z Twojej firmy? Działa tak samo
- Chcesz dynamicznie zmieniać uprawnienia? Mechanizm `create_model` już na to czeka

To jest właśnie różnica między "używaniem AI" a "inżynierią systemów AI". Przejmujesz kontrolę nad procesem decyzyjnym, zamiast oddawać go przypadkowi.

### Co dalej? Faza Execution

Masz w ręku obiekt `ResearchPlan`. To świetnie, ale brutalna prawda jest taka, że to na razie tylko ładnie sformatowana struktura, która  bez wykonania jest bezwartościowa.

W kolejnej lekcji przejdziemy od słów do czynów. Polecą prawdziwe requesty do wyszukiwarki. Zamienimy Twój plan w rzeczywisty ruch sieciowy — wyślemy zapytania do prawdziwych API (lub zasymulujemy realne warunki sieciowe).

Zastanowimy się też, co możemy dodać do odpowiedzi, aby nasz asystent nie był tylko "przekazywaczem linków", lecz robił coś więcej i dostarczał realną wartość.

Przygotuj się, bo wchodzimy w tryb *Execution*.